# UD1 — Semana 3: Consultas sobre Parquet con DuckDB + Benchmark
**Objetivos**: consultar Parquet particionado con DuckDB; comparar tiempos frente a CSV; consolidar conclusiones calidad↔coste.

**RA**: RA1 (b, g), RA4 (f)

In [ ]:

from pathlib import Path
import duckdb, pandas as pd, time, pyarrow.dataset as ds

BASE = Path("..").resolve()
DATA_RAW = BASE/"data/raw"
PARQ_SNAPPY = BASE/"data_curated/parquet/ventas/snappy"
PARQ_ZSTD   = BASE/"data_curated/parquet/ventas/zstd"

con = duckdb.connect()
con.execute("PRAGMA enable_progress_bar;")

# Vista sobre particionado Parquet
con.execute(f"""
CREATE OR REPLACE VIEW ventas_snappy AS 
SELECT * FROM parquet_scan('{PARQ_SNAPPY}/anio=*/mes=*/*.parquet');
""")

# Tres consultas típicas
q1 = "SELECT strftime(fecha, '%Y-%m') AS ym, sum(importe) AS total FROM ventas_snappy GROUP BY 1 ORDER BY 1;"
q2 = "SELECT extract('dow' FROM fecha) AS dow, avg(importe) AS avg_imp FROM ventas_snappy GROUP BY 1 ORDER BY 1;"
q3 = "SELECT * FROM ventas_snappy WHERE importe >= 100 ORDER BY fecha DESC LIMIT 5;"

def timed(sql):
    t0 = time.perf_counter()
    df = con.execute(sql).df()
    t = time.perf_counter()-t0
    return df, round(t, 4)

r1, t1 = timed(q1)
r2, t2 = timed(q2)
r3, t3 = timed(q3)

# Comparativa: leer CSV con pandas y replicar una agregación similar
t0 = time.perf_counter()
ventas_csv = pd.read_csv(DATA_RAW/"ventas.csv", parse_dates=["fecha"])
tmp = ventas_csv.copy()
tmp["ym"] = ventas_csv["fecha"].dt.strftime("%Y-%m")
cmp_csv = tmp.groupby("ym")["importe"].sum().reset_index(name="total")
t_csv = round(time.perf_counter()-t0, 4)

bench = {"duckdb_q1_s": t1, "duckdb_q2_s": t2, "duckdb_q3_s": t3, "pandas_csv_groupby_s": t_csv}
bench


Anota estas cifras en `docs/benchmark_parquet.md` y añade 5–10 líneas de análisis.